# Cosmetic Risk Detector

**Projet final Ironhack — Data Science End-to-End**

Objectif : prédire si un produit contient au moins un ingrédient présent dans la liste de référence utilisée pour construire `has_risk`. Ce label est un proxy documentaire et ne constitue pas une évaluation toxicologique ou médicale.

## 1. Business understanding

Les consommateurs ont besoin d'un premier niveau de lecture des listes INCI. Le projet transforme une liste d'ingrédients en signal d'alerte explicable.

**Question :** peut-on prédire `has_risk` à partir du texte des ingrédients et de la catégorie du produit ?

**Success metrics :** priorité au recall de la classe positive, puis F1, average precision et matrice de confusion.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

df = pd.read_csv("data_products_labeled.csv", low_memory=False)
df.shape

In [ ]:
df[["product_name","ingredients_list","n_ingredients","has_risk"]].head()

In [ ]:
print(df["has_risk"].value_counts())
print(df["has_risk"].value_counts(normalize=True).round(3))
sns.countplot(data=df, x="has_risk")
plt.title("Distribution de la cible has_risk")
plt.show()

## 2. Data preparation

La variable textuelle combine les ingrédients normalisés avec la catégorie, lorsque celle-ci existe. Les colonnes de nutrition sont écartées car elles sont majoritairement vides et peu pertinentes pour une liste INCI.

In [ ]:
df["ingredients_list"] = df["ingredients_list"].fillna("").astype(str)
df["category_text"] = df["categories_fr"].fillna(df["categories"]).fillna("").astype(str)
df["text"] = df["ingredients_list"] + " " + df["category_text"]
df = df[df["text"].str.strip().ne("")].copy()
X = df["text"]
y = df["has_risk"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape

## 3. Baseline et modèle NLP

La baseline prédit toujours la classe majoritaire. Le modèle principal utilise TF-IDF avec des unigrammes et bigrammes, puis une régression logistique pondérée pour tenir compte du déséquilibre modéré.

In [ ]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    print(name)
    print(classification_report(y_test, pred, digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
    print("Average precision:", round(average_precision_score(y_test, proba), 3))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues")
    plt.title(name)
    plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.show()
    return model

baseline = evaluate("Baseline majority", DummyClassifier(strategy="most_frequent"))
model = evaluate("Logistic regression TF-IDF", Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
]))

In [ ]:
# Interprétabilité : termes les plus associés à la classe positive
feature_names = np.array(model.named_steps["tfidf"].get_feature_names_out())
coefs = model.named_steps["clf"].coef_[0]
print("Top termes associés à has_risk=1:")
print(feature_names[np.argsort(coefs)[-20:][::-1]])
print("Top termes associés à has_risk=0:")
print(feature_names[np.argsort(coefs)[:20]])

## 4. Limites et améliorations

- Le label dépend d'une liste de 16 substances de référence : ce n'est pas une mesure complète du risque cosmétique.
- La concentration, le type d'application, la dose et le profil utilisateur ne sont pas disponibles.
- Les ingrédients et les catégories peuvent contenir du bruit ou plusieurs langues.
- Pour une version avancée : validation externe, liste de référence documentée par substance, modèle multi-label par ingrédient, explicabilité par produit et interface Streamlit.